In [ ]:
import requests
from datetime import datetime, timedelta


today = datetime.today()

from_date = (today - timedelta(days=5)).strftime('%Y-%m-%d')
to_date = (today + timedelta(days=5)).strftime('%Y-%m-%d')

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'ko-KR,ko;q=0.9,zh-CN;q=0.8,zh;q=0.7,en-US;q=0.6,en;q=0.5,ja;q=0.4',
    'charset': 'utf-8',
    'origin': 'https://m.sports.naver.com',
    'priority': 'u=1, i',
    'referer': 'https://m.sports.naver.com/kbaseball/schedule/index?category=kbo&date=2026-03-12',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-sports-backend': 'kotlin',
}

league = 'kbo'

league_info = {
    'kbo': {'upperCategoryId': 'kbaseball', 'categoryId': 'kbo'},
    'npb': {'upperCategoryId': 'wbaseball', 'categoryId': 'npb'},
    'mlb': {'upperCategoryId': 'wbaseball', 'categoryId': 'mlb'}
}

params = {
    'fields': 'basic,schedule,baseball,manualRelayUrl',
    'upperCategoryId': league_info[league]['upperCategoryId'],
    'categoryId': league_info[league]['categoryId'],
    'fromDate': from_date,
    'toDate': to_date,
    'roundCodes': '',
    'size': '500',
}


# 디버그 함수, 추후에 지울 것

DEBUG = True

def dprint(*args, **kwargs):
    if DEBUG:
        print(*args, **kwargs)


def safe_int(value, default=0):
    try:
        return int(value)
    except:
        return default


def get_search_end_inning(inn_value):
    """
    예:
    '3'   -> 3
    '3 ⅓' -> 4
    '3 ⅔' -> 4
    '4'   -> 4
    '4 ⅔' -> 5
    """
    if inn_value in [None, '']:
        return 0

    s = str(inn_value).strip()
    s = ' '.join(s.split())

    base_str = s.split()[0]

    try:
        base_inning = int(base_str)
    except:
        return 0

    if ('⅓' in s) or ('⅔' in s) or ('1/3' in s) or ('2/3' in s):
        return base_inning + 1

    return base_inning


def get_result_type(text):
    text = str(text).strip()

    if ('몸에 맞는 볼' in text) or ('사구' in text) or ('몸맞는볼' in text):
        return 'hbp'

    if ('볼넷' in text) or ('고의4구' in text):
        return 'bb'

    return ''


def is_runner_event_text(text):
    text = str(text).strip()
    runner_keywords = ['루주자', '진루', '도루', '폭투', '보크', '견제']
    return any(keyword in text for keyword in runner_keywords)


def make_batter_code_name_map(record_data):
    batter_map = {}

    for side in ['away', 'home']:
        batters = record_data.get('battersBoxscore', {}).get(side, [])
        for batter in batters:
            code = str(
                batter.get('playerCode', batter.get('pcode', batter.get('batterCode', '')))
            ).strip()
            name = batter.get('name', '')
            if code:
                batter_map[code] = name

    return batter_map


def find_final_batter_result(text_options):
    """
    뒤에서부터 보면서
    주자 진루 같은 부가 이벤트를 제외하고
    볼넷/사구 최종 결과를 찾는다.
    """
    if not isinstance(text_options, list) or not text_options:
        return None

    for option in reversed(text_options):
        text = str(option.get('text', '')).strip()
        current_game_state = option.get('currentGameState', {})

        pitcher = str(current_game_state.get('pitcher', '')).strip()
        batter = str(current_game_state.get('batter', '')).strip()

        if not text:
            continue
        if is_runner_event_text(text):
            continue
        if not pitcher or not batter:
            continue

        result_type = get_result_type(text)
        if result_type in ['bb', 'hbp']:
            return {
                'result_type': result_type,
                'text': text,
                'pitcher': pitcher,
                'batter': batter
            }

    return None


def extract_walks_from_one_inning(
    relay_data,
    inning,
    away_starter_pcode,
    home_starter_pcode,
    batter_code_name_map
):
    away_walks = []
    home_walks = []

    text_relays = relay_data.get('result', {}).get('textRelayData', {}).get('textRelays', [])

    for relay in text_relays:
        title = str(relay.get('title', '')).strip()
        text_options = relay.get('textOptions', [])

        final_result = find_final_batter_result(text_options)
        if not final_result:
            continue

        pitcher_code = str(final_result.get('pitcher', '')).strip()
        batter_code = str(final_result.get('batter', '')).strip()
        text = str(final_result.get('text', '')).strip()
        result_type = final_result.get('result_type', '')

        batter_name = batter_code_name_map.get(batter_code, '')

        row = {
            'inning': inning,
            'pitcher_code': pitcher_code,
            'batter_code': batter_code,
            'batter_name': batter_name,
            'title': title,
            'text': text,
            'result_type': result_type
        }

        if pitcher_code == str(away_starter_pcode):
            away_walks.append(row)

        elif pitcher_code == str(home_starter_pcode):
            home_walks.append(row)

    return away_walks, home_walks


def is_inning_start_title(title):
    """
    예: '2회초 한화 공격', '3회말 LG 공격'
    """
    title = str(title).strip()
    return ('회초' in title or '회말' in title) and ('공격' in title)


def get_first_pitch_result_of_first_batter(relay_data, inning, batter_code_name_map):
    """
    각 이닝 relay에서
    '공격 시작 안내' 바로 다음 타자의 첫 공 결과(pitchNum == 1) 추출
    """
    text_relays = relay_data.get('result', {}).get('textRelayData', {}).get('textRelays', [])

    if not text_relays or len(text_relays) < 2:
        return None

    start_idx = None

    for idx in range(len(text_relays) - 1, -1, -1):
        title = str(text_relays[idx].get('title', '')).strip()
        if is_inning_start_title(title):
            start_idx = idx
            break

    if start_idx is None:
        return None

    batter_idx = start_idx - 1
    if batter_idx < 0:
        return None

    first_batter_relay = text_relays[batter_idx]
    title = str(first_batter_relay.get('title', '')).strip()
    text_options = first_batter_relay.get('textOptions', [])

    if not isinstance(text_options, list) or not text_options:
        return None

    for option in text_options:
        if option.get('pitchNum') == 1:
            current_game_state = option.get('currentGameState', {})
            batter_code = str(current_game_state.get('batter', '')).strip()
            batter_name = batter_code_name_map.get(batter_code, '')

            return {
                'inning': inning,
                'title': title,
                'batter_code': batter_code,
                'batter_name': batter_name,
                'pitchResult': option.get('pitchResult', ''),
                'text': option.get('text', ''),
                'stuff': option.get('stuff', ''),
                'speed': option.get('speed', '')
            }

    return None


def extract_inning_results(player_dict, max_inning=25):
    """
    inn1 ~ inn25 중 빈값이 아닌 것만 dict로 반환
    예:
    {'inn1': '우안', 'inn3': '삼진', 'inn7': '4구'}
    """
    inning_results = {}

    for i in range(1, max_inning + 1):
        key = f'inn{i}'
        value = str(player_dict.get(key, '')).strip()

        if value != '':
            inning_results[key] = value

    return inning_results


def extract_batters_boxscore_rows(record_data, game_date, game_id, team_type, team_name):
    """
    battersBoxscore의 away/home 전체 선수 기본 스탯을 row 단위로 추출
    """
    rows = []
    batters = record_data.get('battersBoxscore', {}).get(team_type, [])

    for batter in batters:
        row = {
            'game_date': game_date,
            'game_id': game_id,
            'team_type': team_type,
            'team_name': team_name,

            'playerCode': batter.get('playerCode', ''),
            'name': batter.get('name', ''),
            'pos': batter.get('pos', ''),
            'batOrder': batter.get('batOrder', ''),
            'hasPlayerEnd': batter.get('hasPlayerEnd', ''),

            'ab': batter.get('ab', ''),
            'bb': batter.get('bb', ''),
            'hit': batter.get('hit', ''),
            'kk': batter.get('kk', ''),
            'hr': batter.get('hr', ''),
            'rbi': batter.get('rbi', ''),
            'run': batter.get('run', ''),
            'sb': batter.get('sb', ''),
            'hra': batter.get('hra', ''),

            'inning_results': extract_inning_results(batter)
        }

        rows.append(row)

    return rows


def extract_batter_inning_event_rows(record_data, game_date, game_id, team_type, team_name):
    """
    inn1 ~ inn25의 빈칸이 아닌 값만 개별 행으로 길게 펼쳐서 추출
    """
    rows = []
    batters = record_data.get('battersBoxscore', {}).get(team_type, [])

    for batter in batters:
        for i in range(1, 26):
            key = f'inn{i}'
            value = str(batter.get(key, '')).strip()

            if value == '':
                continue

            rows.append({
                'game_date': game_date,
                'game_id': game_id,
                'team_type': team_type,
                'team_name': team_name,

                'playerCode': batter.get('playerCode', ''),
                'name': batter.get('name', ''),
                'pos': batter.get('pos', ''),
                'batOrder': batter.get('batOrder', ''),

                'inning': i,
                'inning_key': key,
                'result': value
            })

    return rows

def extract_team_batting_total_rows(record_data, game_date, game_id, away_team, home_team):
    """
    battersBoxscore의 awayTotal, homeTotal 팀 합계 스탯 추출
    """
    rows = []
    batters_boxscore = record_data.get('battersBoxscore', {})

    away_total = batters_boxscore.get('awayTotal', {})
    home_total = batters_boxscore.get('homeTotal', {})

    if away_total:
        rows.append({
            'game_date': game_date,
            'game_id': game_id,
            'team_type': 'away',
            'team_name': away_team,
            'opponent_team': home_team,
            'ab': away_total.get('ab', ''),
            'hit': away_total.get('hit', ''),
            'hra': away_total.get('hra', ''),
            'rbi': away_total.get('rbi', ''),
            'run': away_total.get('run', ''),
            'sb': away_total.get('sb', '')
        })

    if home_total:
        rows.append({
            'game_date': game_date,
            'game_id': game_id,
            'team_type': 'home',
            'team_name': home_team,
            'opponent_team': away_team,
            'ab': home_total.get('ab', ''),
            'hit': home_total.get('hit', ''),
            'hra': home_total.get('hra', ''),
            'rbi': home_total.get('rbi', ''),
            'run': home_total.get('run', ''),
            'sb': home_total.get('sb', '')
        })

    return rows



response = requests.get(
    'https://api-gw.sports.naver.com/schedule/games',
    params=params,
    headers=headers
)

games = response.json()['result']['games']

all_rows = []
first_pitch_rows = []
all_batter_rows = []
all_batter_event_rows = []
all_team_total_rows = []

for game in games:
    game_id = game.get('gameId', '')
    if game_id != '20260329LTSS02026':
        continue
    game_date = game.get('gameDate', '')
    home_team = game.get('homeTeamName', '')
    away_team = game.get('awayTeamName', '')
    home_starting = game.get('homeStarterName', '')
    away_starting = game.get('awayStarterName', '')
    status_code = str(game.get('statusCode', '')).strip()
    round_code = str(game.get('roundCode', '')).strip().lower()


    dprint('=' * 80)
    dprint('game_id     :', game_id)
    dprint('game_date   :', game_date)
    dprint('away_team   :', away_team)
    dprint('home_team   :', home_team)
    dprint('away_start  :', away_starting)
    dprint('home_start  :', home_starting)
    dprint('status_code :', status_code)
    dprint('round_code  :', round_code)


    if status_code != 'RESULT':
        dprint('RESULT 경기 아님 -> 스킵')
        continue

    if round_code == 'kbo_e':
        dprint('시범경기(kbo_e) -> 스킵')
        continue

    # 시범경기 제외
    # if round_code == 'kbo_e':
    #     continue
    # 끝난경기만 포함
    # if status_code != 'RESULT':
    #     continue

    record_url = f'https://api-gw.sports.naver.com/schedule/games/{game_id}/record'
    record_response = requests.get(record_url, headers=headers)
    record_json = record_response.json()

    if 'result' not in record_json or 'recordData' not in record_json['result']:
        continue

    record_data = record_json['result']['recordData']

    dprint('record_json keys:', record_json.keys())
    dprint('result keys:', record_json.get('result', {}).keys())
    dprint('record_data keys:', record_data.keys())

    # 팀 타격 합계 row (awayTotal, homeTotal)
    team_total_rows = extract_team_batting_total_rows(
        record_data=record_data,
        game_date=game_date,
        game_id=game_id,
        away_team=away_team,
        home_team=home_team
    )

    all_team_total_rows.extend(team_total_rows)

    # 타자 기본 스탯 row
    away_batter_rows = extract_batters_boxscore_rows(
        record_data=record_data,
        game_date=game_date,
        game_id=game_id,
        team_type='away',
        team_name=away_team
    )

    home_batter_rows = extract_batters_boxscore_rows(
        record_data=record_data,
        game_date=game_date,
        game_id=game_id,
        team_type='home',
        team_name=home_team
    )

    all_batter_rows.extend(away_batter_rows)
    all_batter_rows.extend(home_batter_rows)

    # 타자 이닝별 이벤트 row
    away_batter_event_rows = extract_batter_inning_event_rows(
        record_data=record_data,
        game_date=game_date,
        game_id=game_id,
        team_type='away',
        team_name=away_team
    )

    home_batter_event_rows = extract_batter_inning_event_rows(
        record_data=record_data,
        game_date=game_date,
        game_id=game_id,
        team_type='home',
        team_name=home_team
    )

    all_batter_event_rows.extend(away_batter_event_rows)
    all_batter_event_rows.extend(home_batter_event_rows)


    away_pitchers = record_data.get('pitchersBoxscore', {}).get('away', [])
    home_pitchers = record_data.get('pitchersBoxscore', {}).get('home', [])

    dprint('away_pitchers len:', len(away_pitchers))
    dprint('home_pitchers len:', len(home_pitchers))
    dprint('away_pitchers[0]:', away_pitchers[0] if away_pitchers else '없음')
    dprint('home_pitchers[0]:', home_pitchers[0] if home_pitchers else '없음')

    if not away_pitchers or not home_pitchers:
        continue

    away_pitcher = away_pitchers[0]
    home_pitcher = home_pitchers[0]

    kor_away_pitcher_record = {
        'name': away_pitcher.get('name', away_starting),
        'inn': away_pitcher.get('inn', ''),
        'hit': away_pitcher.get('hit', ''),
        'r': away_pitcher.get('r', ''),
        'kk': away_pitcher.get('kk', ''),
        'bb': away_pitcher.get('bb', ''),
        'ab': away_pitcher.get('ab', ''),
        'bf': away_pitcher.get('bf', ''),
        'era': away_pitcher.get('era', ''),
        'w_l': away_pitcher.get('wls', ''),
        'pcode': str(away_pitcher.get('pcode', away_pitcher.get('playerCode', ''))).strip()
    }

    kor_home_pitcher_record = {
        'name': home_pitcher.get('name', home_starting),
        'inn': home_pitcher.get('inn', ''),
        'hit': home_pitcher.get('hit', ''),
        'r': home_pitcher.get('r', ''),
        'kk': home_pitcher.get('kk', ''),
        'bb': home_pitcher.get('bb', ''),
        'ab': home_pitcher.get('ab', ''),
        'bf': home_pitcher.get('bf', ''),
        'era': home_pitcher.get('era', ''),
        'w_l': home_pitcher.get('wls', ''),
        'pcode': str(home_pitcher.get('pcode', home_pitcher.get('playerCode', ''))).strip()
    }

    away_starter_pcode = kor_away_pitcher_record['pcode']
    home_starter_pcode = kor_home_pitcher_record['pcode']

    if not away_starter_pcode or not home_starter_pcode:
        continue

    away_search_end_inning = get_search_end_inning(kor_away_pitcher_record['inn'])
    home_search_end_inning = get_search_end_inning(kor_home_pitcher_record['inn'])
    max_search_inning = max(away_search_end_inning, home_search_end_inning)

    if max_search_inning == 0:
        continue

    batter_code_name_map = make_batter_code_name_map(record_data)

    away_walks_all = []
    home_walks_all = []

    print('=' * 100)
    print(f'{game_date}  {away_team} vs {home_team}')
    print()

    for inning in range(1, max_search_inning + 1):
        relay_url = f'https://api-gw.sports.naver.com/schedule/games/{game_id}/relay?inning={inning}'
        relay_response = requests.get(relay_url, headers=headers)

        try:
            relay_data = relay_response.json()
        except:
            continue

        dprint('inning:', inning)
        dprint('relay result keys:', relay_data.get('result', {}).keys())
        dprint('textRelayData keys:', relay_data.get('result', {}).get('textRelayData', {}).keys())

        text_relays = relay_data.get('result', {}).get('textRelayData', {}).get('textRelays', [])
        dprint('text_relays len:', len(text_relays))

        for idx, relay in enumerate(text_relays[:5]):
            dprint(f'text_relays[{idx}] title:', relay.get('title', ''))

        if inning == 1:
            dprint('----- relay titles / textOptions raw -----')
            for relay in text_relays:
                title = str(relay.get('title', '')).strip()
                text_options = relay.get('textOptions', [])

                dprint(f'[title] {title}')
                for option in text_options:
                    dprint('   text:', option.get('text', ''))

        away_walks, home_walks = extract_walks_from_one_inning(
            relay_data=relay_data,
            inning=inning,
            away_starter_pcode=away_starter_pcode,
            home_starter_pcode=home_starter_pcode,
            batter_code_name_map=batter_code_name_map
        )

        away_walks_all.extend(away_walks)
        home_walks_all.extend(home_walks)

        first_pitch_info = get_first_pitch_result_of_first_batter(
            relay_data=relay_data,
            inning=inning,
            batter_code_name_map=batter_code_name_map
        )

        if first_pitch_info:
            print(
                f'{inning}회 첫타자 첫공 -> '
                f'{first_pitch_info["batter_name"]} / '
                f'{first_pitch_info["pitchResult"]} / '
                f'{first_pitch_info["text"]}'
            )

            first_pitch_rows.append({
                'game_date': game_date,
                'game_id': game_id,
                'inning': inning,
                'away_team': away_team,
                'home_team': home_team,
                'batter_name': first_pitch_info['batter_name'],
                'batter_code': first_pitch_info['batter_code'],
                'title': first_pitch_info['title'],
                'pitchResult': first_pitch_info['pitchResult'],
                'text': first_pitch_info['text'],
                'stuff': first_pitch_info['stuff'],
                'speed': first_pitch_info['speed']
            })

    away_official_bbhb = safe_int(kor_away_pitcher_record['bb'], 0)
    home_official_bbhb = safe_int(kor_home_pitcher_record['bb'], 0)

    away_walks_all = away_walks_all[:away_official_bbhb]
    home_walks_all = home_walks_all[:home_official_bbhb]

    print()
    print(f'[원정 선발] {kor_away_pitcher_record["name"]}')
    print(f'이닝: {kor_away_pitcher_record["inn"]} / 공식 BB: {kor_away_pitcher_record["bb"]}')
    if away_walks_all:
        for row in away_walks_all:
            print(f'  {row["inning"]}회 - {row["batter_name"]} - {row["result_type"]} - {row["text"]}')
            all_rows.append({
                'game_date': game_date,
                'game_id': game_id,
                'team_type': 'away',
                'pitcher_name': kor_away_pitcher_record['name'],
                'pitcher_pcode': away_starter_pcode,
                'pitcher_inn': kor_away_pitcher_record['inn'],
                'pitcher_bb': kor_away_pitcher_record['bb'],
                'walk_inning': row['inning'],
                'batter_name': row['batter_name'],
                'batter_code': row['batter_code'],
                'result_type': row['result_type'],
                'text': row['text'],
                'away_team': away_team,
                'home_team': home_team
            })
    else:
        print('  없음')

    print(f'추출 BB/HBP: {len(away_walks_all)}')
    print()

    print(f'[홈 선발] {kor_home_pitcher_record["name"]}')
    print(f'이닝: {kor_home_pitcher_record["inn"]} / 공식 BB: {kor_home_pitcher_record["bb"]}')
    if home_walks_all:
        for row in home_walks_all:
            print(f'  {row["inning"]}회 - {row["batter_name"]} - {row["result_type"]} - {row["text"]}')
            all_rows.append({
                'game_date': game_date,
                'game_id': game_id,
                'team_type': 'home',
                'pitcher_name': kor_home_pitcher_record['name'],
                'pitcher_pcode': home_starter_pcode,
                'pitcher_inn': kor_home_pitcher_record['inn'],
                'pitcher_bb': kor_home_pitcher_record['bb'],
                'walk_inning': row['inning'],
                'batter_name': row['batter_name'],
                'batter_code': row['batter_code'],
                'result_type': row['result_type'],
                'text': row['text'],
                'away_team': away_team,
                'home_team': home_team
            })
    else:
        print('  없음')

    print(f'추출 BB/HBP: {len(home_walks_all)}')
    print()


# 필요하면 확인용
print('기본 타자행 수:', len(all_batter_rows))
print('타자 이닝이벤트 행 수:', len(all_batter_event_rows))
print('팀 타격합계 행 수:', len(all_team_total_rows))
print('첫타자 첫공 행 수:', len(first_pitch_rows))
print('투수 BB/HBP 행 수:', len(all_rows))

game_id     : 20260329LTSS02026
game_date   : 2026-03-29
away_team   : 롯데
home_team   : 삼성
away_start  : 비슬리
home_start  : 최원태
status_code : RESULT
round_code  : kbo_r
record_json keys: dict_keys(['code', 'success', 'result'])
result keys: dict_keys(['recordData'])
record_data keys: dict_keys(['etcRecords', 'teamPitchingBoxscore', 'homeTeamNextGames', 'gameInfo', 'awayStandings', 'awayTeamNextGames', 'pitchingResult', 'pitchersBoxscore', 'battersBoxscore', 'todayKeyStats', 'games', 'scoreBoard', 'recentVsGames', 'currentInning', 'homeStandings'])
away_pitchers len: 5
home_pitchers len: 5
away_pitchers[0]: {'bb': 1, 'kk': 5, 'seasonLose': 0, 'ab': 17, 'gameCount': 1, 'bf': 91, 'seasonWin': 0, 'pcode': '56523', 'bbhp': 3, 'inn': '5', 'hr': 0, 'l': 0, 'er': 0, 'hasPlayerEnd': True, 'tb': 'T', 'pa': 20, 'hit': 2, 'r': 1, 's': 0, 'era': '0.00', 'w': 1, 'name': '비슬리', 'wls': '승'}
home_pitchers[0]: {'bb': 2, 'kk': 5, 'seasonLose': 0, 'ab': 21, 'gameCount': 1, 'bf': 83, 'seasonWin': 0, 'pcode'